# Milestone 4 — Heavy Equipment Selling Price Prediction
Working notebook: split, preprocessing, tuning, and model comparisons.

## Setup

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")
df.head()


/tmp/ipykernel_16/2501757741.py:13: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")


,TransactionID,TargetValue,AssetID,ProductConfigID,DataOriginCode,VendorPartnerID,ManufactureYear,OperationalHoursMeter,UtilizationTier,TransactionDate,...,col20,col21,col22,col23,col24,col25,col27,col28,col29,col30
0,1139309,57000.0,117665,88,ACH138,GTX,1997,4640.0,Low,2005-03-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139316,26500.0,1001282,4616,ACH138,GTX,2005,508.0,Low,2009-12-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1139317,21000.0,772709,1948,ACH138,GTX,1994,11540.0,High,2005-08-26,...,Steel,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN
3,1139322,27000.0,902010,3550,ACH138,GTX,2002,4883.0,High,2006-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139333,21500.0,1036259,36014,ACH138,GTX,2009,302.0,Low,2010-08-27,...,Rubber,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN


## Q1 — Stratified vs. Non-Stratified Split

In [2]:
df["UtilizationTier"] = df["UtilizationTier"].fillna("Unknown")

# create price bands
df["PriceBand"] = pd.qcut(df["TargetValue"], q=5, labels=False)

# A) stratified split
train_strat, valid_strat = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df["PriceBand"]
)
counts_strat = train_strat["PriceBand"].value_counts().sort_index().to_numpy()
counts_strat_valid = valid_strat["PriceBand"].value_counts().sort_index().to_numpy()

# B) non-stratified split
train_nostrat, valid_nostrat = train_test_split(
    df, test_size=0.3, random_state=42, stratify=None
)
counts_nostrat_valid = valid_nostrat["PriceBand"].value_counts().sort_index().to_numpy()

# C) compare proportions
prop_strat = counts_strat_valid / counts_strat_valid.sum()
prop_nostrat = counts_nostrat_valid / counts_nostrat_valid.sum()

max_diff = np.max(np.abs(prop_strat - prop_nostrat))
print("Q1 answer:", round(max_diff, 4))


Q1 answer: 0.0033


## Q2 — Preprocessing Pipeline

In [3]:
cat_cols = ["UtilizationTier", "DrivetrainType", "CabinType", "AssetScaleFactor"]
num_cols = ["ManufactureYear", "OperationalHoursMeter"]

X_train = train_strat[cat_cols + num_cols]
X_valid = valid_strat[cat_cols + num_cols]
y_train = train_strat["TargetValue"]
y_valid = valid_strat["TargetValue"]

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("cat", cat_pipeline, cat_cols),
    ("num", num_pipeline, num_cols)
], remainder="drop")

X_train_proc = preprocessor.fit_transform(X_train)
X_valid_proc = preprocessor.transform(X_valid)

total_sum = X_train_proc.sum()
print("Q2 answer:", round(total_sum, 3))
print("N features:", X_train_proc.shape[1])


Q2 answer: 388360.0
N features: 23


## Q3 — Hyperparameter Tuning with RandomizedSearchCV

In [4]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15]
}

rf = RandomForestRegressor(random_state=42)

search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=5,
    cv=3,
    random_state=42,
    n_jobs=-1
)
search.fit(X_train_proc, y_train)

print("Best params:", search.best_params_)
print("Q3 answer (best n_estimators):", search.best_params_["n_estimators"])


Best params: {'n_estimators': 200, 'max_depth': 15}
Q3 answer (best n_estimators): 200


## Q4 — AdaBoost Estimator Error Variance

In [5]:
ada = AdaBoostRegressor(n_estimators=50, random_state=42)
ada.fit(X_train_proc, y_train)

errors = ada.estimator_errors_
variance = np.var(errors)
print("Q4 answer:", round(variance, 4))


Q4 answer: 0.0777


## Q5 — Feature Importance

In [6]:
rf5 = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf5.fit(X_train_proc, y_train)

importances = rf5.feature_importances_
max_idx = np.argmax(importances)
print("Q5 answer (max importance index):", max_idx)


Q5 answer (max importance index): 13


## Q6 — Counting MLP Weights

In [7]:
N = X_train_proc.shape[1]

weights = (N * 128) + (128 * 64) + (64 * 32) + (32 * 1)
print("Q6 answer:", weights)


Q6 answer: 13216


## Q7 — Training Loss After 5 Iterations

In [8]:
mlp7 = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    max_iter=5,
    batch_size=32,
    random_state=42
)
mlp7.fit(X_train_proc, y_train)

print("Q7 answer:", round(mlp7.loss_, 4))


Q7 answer: 221235391.1265


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


## Q8 — Effect of Regularization (alpha)

In [9]:
mlp_a = MLPRegressor(hidden_layer_sizes=(100,), max_iter=5, alpha=0.0001, random_state=42)
mlp_a.fit(X_train_proc, y_train)
mae_a = mean_absolute_error(y_train, mlp_a.predict(X_train_proc))

mlp_b = MLPRegressor(hidden_layer_sizes=(100,), max_iter=5, alpha=1.0, random_state=42)
mlp_b.fit(X_train_proc, y_train)
mae_b = mean_absolute_error(y_train, mlp_b.predict(X_train_proc))

diff = abs(mae_a - mae_b)
print("Q8 answer:", round(diff, 4))


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


Q8 answer: 2.1609


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


## Q9 — Validation Performance

In [10]:
val_preds = mlp7.predict(X_valid_proc)
mae_valid = mean_absolute_error(y_valid, val_preds)
print("Q9 answer:", round(mae_valid, 4))


Q9 answer: 15735.3515
